# B2-022 — Practice p20

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** variational-autoencoder

## Task

Train a tiny VAE on the seeded `mixture2d` data, then sample from its prior and interpolate in latent space with shape and statistic probes.

**Data.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments; they return fresh float32 tensors `latent_data.train_tensor("mixture2d")` of shape `(48,2)` and `latent_data.heldout_tensor("mixture2d")` of shape `(12,2)`, built from the immutable `latent_data.TRAIN_IDS["mixture2d"]` and `latent_data.HELDOUT_IDS["mixture2d"]` in stored order. Pass training rows to the model exactly as returned (same float32 values; no normalization, noise, shuffling, or concatenation), so every row the model's `forward` receives during training hashes, with `latent_data.row_sha256`, into the training part of `latent_data.ROW_SHA256["mixture2d"]`. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a hyperparameter choice. (Here `heldout_rows()` is used only to choose interpolation endpoints after training.)

**Model and loss.** The p19 `TinyVAE` with `TinyVAE(in_dim=2, hidden=16, latent=2)`, same attribute order and methods, constructed immediately after `torch.manual_seed(SEED)`; the p12/p19 `negative_elbo`.

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_vae_for_sampling(model, batch, optimizer)` with the p19 protocol (one `eps_generator = torch.Generator().manual_seed(SEED)` created inside, `eps` of shape `(batch.shape[0], 2)` drawn before each update, $\beta=1$) but **exactly 800** updates; it returns a list of 800 Python floats (the batch-mean total of each update). Call it as `trace = train_vae_for_sampling(model, train_rows(), optimizer)`. Certify that the mean of the last 20 totals is `<= 0.8 *` the first total and that at least one parameter changed.

**Sampling.** Write `sample_prior(model, n, generator)` returning `model.decode(torch.randn(n, 2, generator=generator))` computed under `torch.no_grad()`. Call it with `n = 512` and `torch.Generator().manual_seed(SEED + 2)`. Probes, with `T = train_rows()`:
1. shape `(512,2)`, dtype float32, all finite, `requires_grad` false;
2. every coordinate of the sample mean within `0.5` (absolute) of `T.mean(dim=0)`;
3. every coordinate of the sample standard deviation at least `0.5 *` the corresponding `T.std(dim=0)`;
4. assigning each sample to the nearest of the generator's literal component means `(-2, 0)`, `(2, 0)`, `(0, 2.5)` (Euclidean distance), every component receives at least `51` samples.

**Interpolation.** Write `interpolate_latents(model, x_a, x_b, steps)` that, under `torch.no_grad()`, takes the posterior means `mu_a, _ = model.encode(x_a)` and `mu_b, _ = model.encode(x_b)` of two `(1,2)` rows, builds `t = torch.linspace(0, 1, steps).unsqueeze(1)` and `z_path = (1 - t) * mu_a + t * mu_b`, and returns `(z_path, model.decode(z_path))`. Use `x_a = heldout_rows()[0:1]`, `x_b = heldout_rows()[1:2]`, `steps = 9`. Probes (`ATOL = RTOL = 1e-6`): both outputs have shape `(9,2)`; `z_path[0]` equals `mu_a[0]` and `z_path[-1]` equals `mu_b[0]`; all eight consecutive differences of `z_path` are equal; the decoded first and last rows equal `model.decode(mu_a)[0]` and `model.decode(mu_b)[0]`.

Finally, in three sentences, explain why probes 2–4 use tolerances while the interpolation probes are exact, and what a collapsed model would show in each.

**Required answer-check assertions.** The training certificate, sampling probes 1–4, and every interpolation probe.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

COMPONENT_MEANS = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.5]])
N_PRIOR_SAMPLES = 512
INTERPOLATION_STEPS = 9
ATOL = 1e-6
RTOL = 1e-6

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.